# Subject Dependent Experiments
model trained on one individual subject's samples, shuffled and divided into an 80-20 train-test split

In [1]:
import numpy as np
import tensorflow as tf
from Dataloader import read_excel_eeg, read_excel_nirs
from Model_MI import eeg_fnirs_cnn_v2

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, confusion_matrix
import os
import pandas as pd
import openpyxl

# data root
dir_path_f = "../../Data/EF-MI-MA/"
dir_path_e = "../../Data/EF-MI-MA/"

# result logging
if not os.path.exists('Results'):
	os.makedirs('Results')
result_file = "Results/Log.txt"
log_excel = 'Results/Log.xlsx'

# hyperparameters
baseline_model=False # EF-Net or ResNet (basleine)
resnet=False
if baseline_model:
	lr=5e-5
	EPOCHS=50
else:
	lr=1e-3
	EPOCHS=200
BATCH_SIZE = 16
RANDOM_STATE = 38
n_subject = 26
mode = 0

acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
for subject in range(n_subject):
	subject += 1
	# subject = f'{subject:02d}'
	X_eeg, Y = read_excel_eeg(subject,mode)
	X_fnirs = read_excel_nirs(subject, mode)

	X_eeg = X_eeg[:, ::4, :, :]
	X_fnirs = X_fnirs[:, ::4, :, :]

	# Standardize our signals data
	# Reshaping the data samples into 2D applying the scaling and then reshaping back
	scaler = StandardScaler()
	XeegS_3D = scaler.fit_transform(X_eeg.reshape(-1, X_eeg.shape[-1])).reshape(X_eeg.shape)
	XfnirsS_3D = scaler.fit_transform(X_fnirs.reshape(-1, X_fnirs.shape[-1])).reshape(X_fnirs.shape)

	# adjustment to labels to make compatible with binary_crossentropy
	# Y=Y[:,0]

	print(XeegS_3D.shape)
	print(XfnirsS_3D.shape)
	print(Y.shape)

   #  # reshape data
   #  if baseline_model:
   #      XeegS_3D=XeegS
   #      XfnirsS_3D=XfnirsS
   #  else:
   #      # introduce an extra dimension
   #      XeegS_3D=np.reshape(XeegS,(XeegS.shape[0],XeegS.shape[1],XeegS.shape[2],1))
   #      XfnirsS_3D=np.reshape(XfnirsS,(XfnirsS.shape[0],XfnirsS.shape[1],XfnirsS.shape[2],1))
   # pirnt(XeegS_3D.shape)

	# split data into training and testing sets (60/40 ratio)
	Xeeg_train, Xeeg_test, y_train, y_test = train_test_split(XeegS_3D, Y, test_size=0.2,random_state=RANDOM_STATE)
	print(Xeeg_train.shape,Xeeg_test.shape)

	Xfnirs_train, Xfnirs_test, y_train2, y_test2 = train_test_split(XfnirsS_3D, Y, test_size=0.2, random_state=RANDOM_STATE)
	print(Xfnirs_train.shape, Xfnirs_test.shape)

	if baseline_model:
		hybridmodel = eeg_fnirs_cnn_v2()
		# hybridmodel= models.eeg_fnirs_vgg(Xeeg_train,Xfnirs_train, resnet)
	else:
		hybridmodel = eeg_fnirs_cnn_v2()
	hybridmodel.summary()
	hybridmodel.compile(optimizer=Adam(learning_rate=lr),loss='binary_crossentropy',metrics=['accuracy'])

	# save weights from the highest accuracy epoch
	checkpoint_filepath = '/tmp/checkpoint.weights.h5'
	model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
		filepath=checkpoint_filepath,
		save_weights_only=True,
		monitor='val_accuracy',
		mode='max',
		save_best_only=True,
		verbose=1)

	hybridmodel.fit(x=[Xeeg_train, Xfnirs_train], y=y_train, validation_data=([Xeeg_test, Xfnirs_test], y_test), epochs=EPOCHS, callbacks=[model_checkpoint_callback])
	
	hybridmodel.load_weights(checkpoint_filepath)
	probabilities = hybridmodel.predict([Xeeg_test, Xfnirs_test])
	y_preds= np.round(probabilities)
	loss,accuracy = hybridmodel.evaluate([Xeeg_test,Xfnirs_test], y_test)

	accuracy = accuracy_score(y_test, y_preds)
	precision = precision_score(y_test, y_preds)
	recall = recall_score(y_test, y_preds)
	f1 = f1_score(y_test, y_preds)
	kappa = cohen_kappa_score(y_test, y_preds)

	# acc_list.append(accuracy)
	# precision_list.append(precision)
	# recall_list.append(recall)
	# f1_list.append(f1)
	# kappa_list.append(kappa)

	# log_path = f'Results/log_s{subject:2d}.txt'
	# with open(log_path, 'a') as log_file:
	# 	log_file.write(f'Accuracy: {accuracy:.2f}\n')
	# 	log_file.write(f'Precision: {precision:.2f}\n')
	# 	log_file.write(f'Recall: {recall:.2f}\n')
	# 	log_file.write(f'F1: {f1:.2f}\n')
	# 	log_file.write(f'Kappa: {kappa:.2f}\n')
	# 	log_file.write('\n')

	new_row = pd.DataFrame([[accuracy, precision, recall, f1, kappa]], 
						columns=['Accuracy', 'Precision', 'Recall', 'F1', 'Kappa'])
	new_row = new_row.round(2)
	if not os.path.exists(log_excel):
		new_row.to_excel(log_excel, index=False, engine='openpyxl')
	else:
		with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
			if 'Sheet1' in writer.sheets:
				startrow = writer.sheets['Sheet1'].max_row
			else:
				startrow = 0
			new_row.to_excel(writer, index=False, header=False, startrow=startrow)

2025-05-08 17:35:30.257025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-08 17:35:30.274196: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-08 17:35:30.279493: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-08 17:35:30.293188: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


1 EEG (60, 4000, 30, 1) Labels (60,)
1 fNIRS (60, 200, 36, 1) Labels (60,)
(60, 1000, 30, 1)
(60, 50, 36, 1)
(60,)
(48, 1000, 30, 1) (12, 1000, 30, 1)
(48, 50, 36, 1) (12, 50, 36, 1)


2025-05-08 17:35:56.778159: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 56679 MB memory:  -> device: 0, name: NVIDIA A800 80GB PCIe, pci bus id: 0000:a1:00.0, compute capability: 8.0


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1000, 30,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 994, 30,   │        256 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 988, 30,   │      7,200 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 982, 30,   │      7,200 │ conv2d_1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 140, 30,   │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 50, 36, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 140, 30,   │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 47, 36,    │        160 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 140, 30,   │        128 │ dropout[0][0]     │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 44, 36,    │      4,128 │ conv2d_6[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 137, 27,   │     32,832 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 11, 36,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 134, 24,   │     65,600 │ conv2d_3[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 11, 36,    │          0 │ max_pooling2d_2[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 131, 21,   │     65,600 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 11, 36,    │        128 │ dropout_3[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 5, 64) │          0 │ conv2d_5[0][0]  

...

 Trainable params: 3,502,145 (13.36 MB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/200


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor', 'keras_tensor_1']. Received: the structure of inputs=('*', '*')
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.4844 - loss: 0.6915
Epoch 1: val_accuracy improved from -inf to 0.50000, saving model to /tmp/checkpoint.weights.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 16s 7s/step - accuracy: 0.4896 - loss: 0.6917 - val_accuracy: 0.5000 - val_loss: 0.6977
Epoch 2/200
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.6250 - loss: 0.6876
Epoch 2: val_accuracy did not improve from 0.50000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.6250 - loss: 0.6859 - val_accuracy: 0.5000 - val_loss: 0.6924
Epoch 3/200
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.7812 - loss: 0.6650
Epoch 3: val_accuracy did not improve from 0.50000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.7743 - loss: 0.6667 - val_accuracy: 0.4167 - val_loss: 0.6887
Epoch 4/200
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.7500 - loss: 0.6468
Epoch 4: val_accuracy improved from 0.50000 to 0.66667, saving model to /tmp/checkpoint.weights.h5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step -